# Case
**Creates binary labels in raster format**

The labels are either based on a provided text- or Esri-shapefile.<br>
To spatially align the labels with other data sets, a template raster (base raster) can be provided.


In [15]:
import geopandas as gpd

from pathlib import Path
from rasterio.enums import MergeAlg
    
from rasterio import features

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.utilities.conversions import  create_binary_raster
from beak.utilities.io import load_raster, save_raster
from beak.utilities.raster_processing import snap_raster


**User definitions**

In [20]:
# Set paths
BASE_PATH = files("beak.data")
PATH_SHAPEFILE = files("beak.data") / "RAW" / "boundaries" / "MagmaticNiCo_UpperMidwest" / "focus_areas_MagmaticNiCo_and_TA2_HM6_nat_buffered10000.shp"
BASE_RASTER_NAME = "EPSG_102003_RES_500_NICO_UpperMidwest_CLP_TA2_HM6_NAT.tif"
SNAP_RASTER = files("beak.data") / "BASE_RASTERS" / "EPSG_102003_RES_500_NICO_UpperMidwest.tif"

# Export path
PATH_EXPORT = files("beak.data") / "BASE_RASTERS" / BASE_RASTER_NAME

# Get current working directory
CURRENT_DIR = Path.cwd()

# Overwrite export path with current directory for testing
OUT_FILE = CURRENT_DIR / BASE_RASTER_NAME

# Reference coordinate system code
RESOLUTION = 500

# Select specific features
QUERY = None

# Create Base Raster

## From shapefile

`fill_negatives`<p>
- True will fill the nodata values with a fill value, commonly 0 (i.e., the extent of the raster is filled completeley with values)<p>
- False will only convert the queried polygons to **one** and the rest of the raster will be **nodata**<p>
**Base rasters** commonly have **nodata** (`fill_negatives=False`) instead of filled values

**The existing raster file was created using GIS to ensure the correct selection and snapping of the raster in relation to the BASE_RASTER of the smaller focus area extent.**

In [22]:
# Load shapefile and create geodataframe
gdf = gpd.read_file(PATH_SHAPEFILE)

# Setting OUT_FILE to None to avoid overwriting the original file
save = False

if save:
  base_array = create_binary_raster(gdf, resolution=RESOLUTION, query=QUERY, all_touched=False, same_shape=True, fill_negatives=False, out_file=OUT_FILE)

  # Snap raster to reference
  snapped_raster, snapped_meta = snap_raster(raster=load_raster(OUT_FILE), snap_raster=load_raster(SNAP_RASTER))
  save_raster(path=OUT_FILE, array=snapped_raster, nodata_value=-99, dtype="int8", metadata=snapped_meta)